# meeting_extracted_digest_preview — 月報版面草稿（會議紀錄 × Sales Brief）

- 用途：**跟 user 討論版面用的預覽**，不是正式產出。抓目前鏡頭表 `g_meeting_items_sem_bu`（Stage A 寫的、已 token 化、無人名）裡某客戶 / 應用 / 月份的條目，排成「月報」的樣子用 `displayHTML` 顯示，同時印出 markdown 可貼到文件。Sales Brief 那一欄目前是**示意文字**（W2 對照 §4.3–§4.5 尚未開發），版面定了才知道 W2 要產什麼。
- 輸入：`g_meeting_items_sem_bu`（003 建、Stage A 寫）。只讀不寫。
- 參數（widgets）：`catalog`、`schema`、`unit`、`customer`（entity_id，例 `acer`；空 = 不限）、`app`（TV / NB / MNT；空 = 不限）、`month`（yyyy-MM；空 = 資料最多的那個月）、`max_per_topic`（每主題最多列幾條）、`include_dups`（true 連同文重複一起列）
- 排程：無，手動跑。
- 負責人 / 更新日期：（填）/ 2026-10-19

版面（§1–§7）：資料範圍 → 本月重點（示意） → **主題對照表**（會議說了什麼｜Sales Brief 說了什麼｜狀態） → 事件時間軸 → 背景情報與口徑（候選） → 待確認 → 資料缺口。要改版面改 [c03]，要改主題中文名改 [c02]。


In [ ]:
# [c01] params
# 月報版面預覽：只讀鏡頭表，不寫任何表。由 job parameters 覆蓋；改預設值要先 dbutils.widgets.removeAll()。
import re as _re

dbutils.widgets.text("catalog", "micenter")
dbutils.widgets.text("schema", "mi3_datahub_prod")
dbutils.widgets.text("unit", "bu")
dbutils.widgets.text("customer", "acer")        # entity_id（s_semantic_entity）；空 = 不限客戶
dbutils.widgets.text("app", "MNT")              # TV / NB / MNT；空 = 不限（ALL 的條目一律納入）
dbutils.widgets.text("month", "")               # yyyy-MM；空 = 該客戶 / 應用條目最多的月份
dbutils.widgets.text("max_per_topic", "8")      # 主題對照表每主題最多列幾條
dbutils.widgets.dropdown("include_dups", "false", ["true", "false"])  # true：同文重複（dup_key 非空）也列

cfg = {
    "catalog": dbutils.widgets.get("catalog").strip(),
    "schema": dbutils.widgets.get("schema").strip(),
    "unit": dbutils.widgets.get("unit").strip(),
    "customer": dbutils.widgets.get("customer").strip(),
    "app": dbutils.widgets.get("app").strip().upper(),
    "month": dbutils.widgets.get("month").strip(),
    "max_per_topic": int(dbutils.widgets.get("max_per_topic")),
    "include_dups": dbutils.widgets.get("include_dups") == "true",
}
assert cfg["catalog"] and cfg["schema"] and cfg["unit"], "catalog / schema / unit 不可為空"
assert cfg["app"] in ("", "TV", "NB", "MNT"), "app 只能是 TV / NB / MNT 或空"
assert not cfg["month"] or _re.fullmatch(r"\d{4}-(0[1-9]|1[0-2])", cfg["month"]), "month 格式 yyyy-MM，或留空"
assert 1 <= cfg["max_per_topic"] <= 50, "max_per_topic 1–50"
print(cfg)

In [ ]:
# [c02] imports
# [c02]、[c03] 是純 Python（不碰 spark / dbutils），tests/ 直接載入。主題與陳述類型的中文名在這裡改。
import html
import re
from collections import Counter, defaultdict
from datetime import date

# 鏡頭表要讀的欄（全部非 S3；summary / quote 已 token 化）
LENS_COLS = ["item_key", "pk_seq", "segment_no", "category", "category_rule", "app", "customer", "term", "summary",
             "quote", "date", "source", "statement_type", "topic_tag_primary", "content_type", "sensitivity",
             "confidence", "dup_key", "verified", "as_of"]

# 主題順序 = 月報從上到下的順序（商務在前、工程在後）；不在表裡的標籤排最後
TOPIC_ZH = {
    "demand": "需求 / 預測", "allocation": "Allocation / 限簽", "price": "價格 / 報價", "cost": "成本",
    "product_plan": "產品規劃 / 開案", "spec": "規格", "schedule": "時程", "supply_chain": "供應鏈 / 料況",
    "capacity": "產能 / 廠區", "quality": "品質", "engineering": "工程 / 驗證", "commercial_relation": "商務關係 / 對手",
    "terminology": "術語 / 口徑", "admin": "行政",
}
TOPIC_ORDER = list(TOPIC_ZH)
STATEMENT_ZH = {"fact": "事實", "plan": "規劃", "request": "要求", "decision": "決定", "issue": "問題",
                "opinion": "看法", "definition": "定義"}
CATEGORY_ZH = {"semantic": "口徑 / 定義", "intel": "背景情報", "event": "事件"}
SIDE_ZH = {"auo": "AUO", "customer": "客戶", "third_party": "第三方", "unknown": "—"}
# 參與者表只取這四欄（不碰姓名 / email）；summary 裡的 <P07> 依陣營顯示成「客戶方（acer）」「AUO 方」（D33）
PART_COLS = ["pk_seq", "token", "side", "company"]
TOKEN_RE = re.compile(r"<([PG]\d{2,})>")
SIDE_LABEL = {"auo": "AUO 方", "customer": "客戶方", "third_party": "第三方", "unknown": "與會者"}

# Sales Brief 欄位：W2 對照（§4.3–§4.5）尚未開發，先用示意文字佔位，版面定了再接 g_meeting_sb_reconcile_bu
SB_PLACEHOLDER = "（示意）Sales Brief 本月對此主題的說法；W2 對照開發後由 `g_meeting_sb_reconcile_bu` 填入"
STATUS_PLACEHOLDER = "待對照"
MAX_TIMELINE = 30
MAX_INTEL = 12
MAX_REVIEW = 10
LOW_CONFIDENCE = 0.6

In [ ]:
# [c03] pure_render
# rows（鏡頭表列 dict）→ sections（版面結構）→ markdown / HTML。版面在 build_sections 改；兩個 renderer 只管格式。
# section = {"title", "kind": kv | table | bullets | text, "data"}；table 的 cell 可以是 list（多行）。
_DATE_TAIL_RE = re.compile(r"\s+\d{4}-\d{2}-\d{2}$")


def _s(v) -> str:
    return "" if v is None else str(v)


def subject_of(source: str | None) -> str:
    """source = 'subject segment_date — side' → subject。"""
    head = (source or "").split(" — ")[0]
    return _DATE_TAIL_RE.sub("", head).strip()


def side_of(source: str | None) -> str:
    parts = (source or "").split(" — ")
    return SIDE_ZH.get(parts[1].strip(), parts[1].strip()) if len(parts) > 1 else "—"


def mmdd(d) -> str:
    return f"{d.month:02d}/{d.day:02d}" if isinstance(d, date) else _s(d)


def topic_zh(tag: str | None) -> str:
    return TOPIC_ZH.get(tag or "", tag or "（無主題）")


def token_labels(parts: list[dict]) -> dict[tuple[int, str], str]:
    """participants 列（pk_seq / token / side / company）→ {(pk_seq, token): 顯示標籤}。
    人：客戶方（acer）/ AUO 方 / 第三方（inx）/ 與會者；群組 Gnn：群組（acer）。同 token 多列取第一個有 side 的。"""
    out: dict[tuple[int, str], str] = {}
    for p in parts:
        key = (p.get("pk_seq"), p.get("token"))
        if key in out and out[key] != SIDE_LABEL["unknown"]:
            continue
        side = SIDE_LABEL.get(p.get("side") or "unknown", SIDE_LABEL["unknown"])
        if str(p.get("token") or "").startswith("G"):
            side = "群組"
        company = p.get("company")
        out[key] = f"{side}（{company}）" if company and side not in ("AUO 方", "與會者") else side
    return out


def resolve_tokens(text: str | None, pk_seq, labels: dict | None) -> str:
    """<P07> / <G01> → 陣營標籤；labels 沒有的 token → 與會者。labels None 就原樣。"""
    if not text or labels is None:
        return _s(text)
    return TOKEN_RE.sub(lambda m: labels.get((pk_seq, m.group(1)), SIDE_LABEL["unknown"]), text)


def _with_labels(rows: list[dict], labels: dict | None) -> list[dict]:
    return [dict(r, summary=resolve_tokens(r.get("summary"), r.get("pk_seq"), labels)) for r in rows]


def item_line(r: dict) -> str:
    return f"{mmdd(r.get('date'))} {_s(r.get('summary'))}（{subject_of(r.get('source'))}）"


def build_sections(rows: list[dict], intel_rows: list[dict], cfg: dict, month: str, labels: dict | None = None) -> list[dict]:
    """rows = 該客戶 / 應用 / 月份的條目（已依 include_dups 處理前的全部）；intel_rows = 該客戶 / 應用所有月份的 intel / semantic；
    labels = token_labels 的輸出（None 就讓 <Pnn> 原樣）。"""
    rows, intel_rows = _with_labels(rows, labels), _with_labels(intel_rows, labels)
    n_all = len(rows)
    n_dup = sum(1 for r in rows if r.get("dup_key"))
    kept = rows if cfg.get("include_dups") else [r for r in rows if not r.get("dup_key")]
    who = " ".join(x for x in (cfg.get("customer") or "全部客戶", cfg.get("app") or "全部應用") if x)
    title = f"{who} {month} 會議紀錄 × Sales Brief 月報（版面草稿）"

    by_topic: dict[str, list[dict]] = defaultdict(list)
    for r in kept:
        by_topic[r.get("topic_tag_primary") or ""].append(r)
    topics = sorted(by_topic, key=lambda t: (TOPIC_ORDER.index(t) if t in TOPIC_ORDER else 99))

    meta = [
        ("月份", month),
        ("範圍", who),
        ("來源信件", f"{len({r.get('pk_seq') for r in kept})} 封"),
        ("場合（主旨去重）", f"{len({subject_of(r.get('source')) for r in kept})} 場"),
        ("條目", f"{len(kept)} 條（原始 {n_all}，同文重複 {n_dup} 條{'已併入' if cfg.get('include_dups') else '已略去'}）"),
        ("S2 條目", f"{sum(1 for r in kept if r.get('sensitivity') == 'S2')} 條（對外版要遮）"),
        ("Sales Brief", "尚未接（W2 對照未開發，右欄為示意）"),
    ]

    top = Counter(r.get("topic_tag_primary") or "" for r in kept).most_common(3)
    highlights = [
        f"{topic_zh(t)}：{n} 條，"
        + "、".join(f"{STATEMENT_ZH.get(k, k)} {v}" for k, v in
                    Counter(r.get("statement_type") for r in by_topic[t]).most_common(3))
        for t, n in top
    ] + ["（示意）正式版這段由 LLM 依下表產 3–5 句摘要，並標出與 Sales Brief 不一致的點。"]

    topic_table = {"headers": ["主題", "會議說了什麼", "Sales Brief 說了什麼", "狀態"], "rows": []}
    cap = cfg.get("max_per_topic", 8)
    for t in topics:
        rs = sorted(by_topic[t], key=lambda r: (_s(r.get("date")), _s(r.get("item_key"))))
        lines = [item_line(r) for r in rs[:cap]]
        if len(rs) > cap:
            lines.append(f"…另 {len(rs) - cap} 條")
        topic_table["rows"].append([f"{topic_zh(t)}（{len(rs)}）", lines, SB_PLACEHOLDER, STATUS_PLACEHOLDER])

    timeline = {"headers": ["日期", "類型", "內容", "發言方", "場合"], "rows": []}
    for r in sorted(kept, key=lambda r: (_s(r.get("date")), _s(r.get("item_key"))))[:MAX_TIMELINE]:
        timeline["rows"].append([mmdd(r.get("date")), STATEMENT_ZH.get(r.get("statement_type"), _s(r.get("statement_type"))),
                                 _s(r.get("summary")), side_of(r.get("source")), subject_of(r.get("source"))])

    intel = {"headers": ["類別", "有效基準", "主題", "內容", "PM 審核"], "rows": []}
    for r in sorted(intel_rows, key=lambda r: (r.get("category") != "semantic", _s(r.get("as_of")), _s(r.get("term"))))[:MAX_INTEL]:
        v = r.get("verified")
        intel["rows"].append([CATEGORY_ZH.get(r.get("category"), _s(r.get("category"))), _s(r.get("as_of")) or "—",
                              _s(r.get("term")), _s(r.get("summary")), {"y": "通過", "n": "不採用"}.get(v, "未審")])

    review = {"headers": ["條目", "原因", "內容"], "rows": []}
    for r in kept:
        why = []
        if r.get("category") and r.get("category_rule") and r["category"] != r["category_rule"]:
            why.append(f"分類與先驗不同（{r['category_rule']}→{r['category']}）")
        if (r.get("confidence") is not None) and r["confidence"] < LOW_CONFIDENCE:
            why.append(f"信心 {r['confidence']}")
        if r.get("sensitivity") == "S2":
            why.append("S2 敏感")
        if why:
            review["rows"].append([_s(r.get("item_key"))[:8], "；".join(why), _s(r.get("summary"))])
        if len(review["rows"]) >= MAX_REVIEW:
            break

    gaps = [
        "約 29% 的信（142 封）正文在進資料平台前就被截斷，那些會議不在這份報表裡（ETL 問題，處理中）。",
        "條目的客戶欄約半數由程式補對、部分仍空白；按客戶篩選會漏，正式版前要重萃一次。",
        "附件（Excel / PPT / PDF）與圖片裡的內容不讀；只有信件正文的文字進得來。",
        "同一段紀要在連續幾週的信裡重複出現，已自動合併（上面「同文重複」的數字）。",
        "人名不進報表：條目裡的人以陣營顯示（客戶方（acer）/ AUO 方），對外版再把 S2 條目遮掉。",
    ]
    return [
        {"title": title, "kind": "title", "data": None},
        {"title": "1. 資料範圍", "kind": "kv", "data": meta},
        {"title": "2. 本月重點（示意）", "kind": "bullets", "data": highlights},
        {"title": "3. 主題對照：會議 × Sales Brief", "kind": "table", "data": topic_table},
        {"title": f"4. 事件時間軸（最多 {MAX_TIMELINE} 條）", "kind": "table", "data": timeline},
        {"title": "5. 背景情報與口徑（Stage A 候選，跨月）", "kind": "table", "data": intel},
        {"title": "6. 待確認（分類存疑 / 低信心 / S2）", "kind": "table", "data": review},
        {"title": "7. 資料缺口", "kind": "bullets", "data": gaps},
    ]


# ---------- markdown ----------
def _md_cell(v) -> str:
    if isinstance(v, list):
        return "<br>".join(_md_cell(x) for x in v)
    return _s(v).replace("|", "\\|").replace("\n", " ")


def render_md(sections: list[dict]) -> str:
    out: list[str] = []
    for sec in sections:
        k, d = sec["kind"], sec["data"]
        if k == "title":
            out.append(f"# {sec['title']}\n")
            continue
        out.append(f"## {sec['title']}\n")
        if k == "kv":
            out += [f"- **{a}**：{b}" for a, b in d]
        elif k == "bullets":
            out += [f"- {x}" for x in d]
        elif k == "text":
            out.append(_s(d))
        elif k == "table":
            if not d["rows"]:
                out.append("（無）")
            else:
                out.append("| " + " | ".join(d["headers"]) + " |")
                out.append("|" + "---|" * len(d["headers"]))
                out += ["| " + " | ".join(_md_cell(c) for c in row) + " |" for row in d["rows"]]
        out.append("")
    return "\n".join(out).rstrip() + "\n"


# ---------- HTML（displayHTML 用；不依賴外部套件）----------
_CSS = ("<style>.dg{font-family:system-ui,-apple-system,'Segoe UI','Noto Sans TC',sans-serif;max-width:1200px;line-height:1.5}"
        ".dg h1{font-size:1.4em}.dg h2{font-size:1.1em;margin-top:1.4em;border-bottom:1px solid #ddd}"
        ".dg table{border-collapse:collapse;width:100%;font-size:.92em}.dg th,.dg td{border:1px solid #ccc;padding:4px 8px;"
        "vertical-align:top;text-align:left}.dg th{background:#f3f3f3}.dg td ul{margin:0;padding-left:1.1em}"
        ".dg .ph{color:#888;font-style:italic}</style>")


def _h(v) -> str:
    return html.escape(_s(v))


def _html_cell(v) -> str:
    if isinstance(v, list):
        return "<ul>" + "".join(f"<li>{_h(x)}</li>" for x in v) + "</ul>"
    s = _h(v)
    return f'<span class="ph">{s}</span>' if s.startswith("（示意）") or s == STATUS_PLACEHOLDER else s


def render_html(sections: list[dict]) -> str:
    out = [_CSS, '<div class="dg">']
    for sec in sections:
        k, d = sec["kind"], sec["data"]
        if k == "title":
            out.append(f"<h1>{_h(sec['title'])}</h1>")
            continue
        out.append(f"<h2>{_h(sec['title'])}</h2>")
        if k == "kv":
            out.append("<ul>" + "".join(f"<li><b>{_h(a)}</b>：{_h(b)}</li>" for a, b in d) + "</ul>")
        elif k == "bullets":
            out.append("<ul>" + "".join(f"<li>{_html_cell(x)}</li>" for x in d) + "</ul>")
        elif k == "text":
            out.append(f"<p>{_h(d)}</p>")
        elif k == "table":
            if not d["rows"]:
                out.append("<p>（無）</p>")
            else:
                out.append("<table><tr>" + "".join(f"<th>{_h(h)}</th>" for h in d["headers"]) + "</tr>")
                out += ["<tr>" + "".join(f"<td>{_html_cell(c)}</td>" for c in row) + "</tr>" for row in d["rows"]]
                out.append("</table>")
    out.append("</div>")
    return "\n".join(out)

In [ ]:
# [c10] load_source
# 只讀鏡頭表的 active 列；先過客戶 / 應用，再決定月份（空 = 條目最多的月），collect 該月條目（一客戶一月幾百條級）。
# intel / semantic 另抓同客戶 / 應用的所有月份（給 §5），最多 200 條。
from pyspark.sql import functions as F

fq_lens = f"{cfg['catalog']}.{cfg['schema']}.g_meeting_items_sem_{cfg['unit']}"
assert spark.catalog.tableExists(fq_lens), f"{fq_lens} 不存在：先跑 003 與 meeting_extracted_sem_stage_a"
base = spark.table(fq_lens).where(F.col("active")).select(*LENS_COLS)
if cfg["customer"]:
    base = base.where(F.col("customer") == F.lit(cfg["customer"]))
if cfg["app"]:
    base = base.where(F.col("app").isin(cfg["app"], "ALL"))
base = base.withColumn("_month", F.date_format("date", "yyyy-MM"))

months = (base.where(F.col("_month").isNotNull()).groupBy("_month").count().orderBy(F.desc("count"), F.desc("_month"))
          .limit(24).collect())
print("各月條目數：", {r["_month"]: r["count"] for r in sorted(months, key=lambda r: r["_month"])})
month = cfg["month"] or (months[0]["_month"] if months else None)
assert month, "這個客戶 / 應用沒有任何條目：換 customer / app"

rows = [r.asDict() for r in base.where(F.col("_month") == F.lit(month)).drop("_month").limit(3000).collect()]
intel_rows = [r.asDict() for r in base.where(F.col("category").isin("intel", "semantic") & F.col("dup_key").isNull())
              .drop("_month").limit(200).collect()]
# 參與者表只取 token / side / company（不取姓名），給 [c03] 把 <P07> 顯示成陣營
fq_part = f"{cfg['catalog']}.{cfg['schema']}.s_meeting_participants_{cfg['unit']}"
pks = sorted({r["pk_seq"] for r in rows} | {r["pk_seq"] for r in intel_rows})
part_rows = ([r.asDict() for r in spark.table(fq_part).select(*PART_COLS).where(F.col("pk_seq").isin(*pks)).collect()]
             if pks and spark.catalog.tableExists(fq_part) else [])
labels = token_labels(part_rows)
print(f"month {month} | rows {len(rows)} | intel/semantic（跨月）{len(intel_rows)} | tokens {len(labels)}")

In [ ]:
# [c11] render
# 組版面 → displayHTML 預覽；markdown 全文另印在下面（可整段複製到文件給 user）。不寫表。
sections = build_sections(rows, intel_rows, cfg, month, labels)
html_out, md_out = render_html(sections), render_md(sections)
displayHTML(html_out)
print(f"markdown {len(md_out)} 字：\n")
print(md_out)